In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import roc_curve

In [3]:
PROJECT_ROOT = Path.home() / "Documents" / "interview-anonymization"

VOXCELEB_WAV_DIR = (
    Path.home()
    / "Documents"
    / "voxceleb1"
    / "wav"
)

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"

print("Project root:", PROJECT_ROOT)
print("VoxCeleb wav:", VOXCELEB_WAV_DIR)
print("VoxCeleb exists:", VOXCELEB_WAV_DIR.exists())

Project root: /Users/juni/Documents/interview-anonymization
VoxCeleb wav: /Users/juni/Documents/voxceleb1/wav
VoxCeleb exists: True


In [4]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.ecapa import load_ecapa, extract_embedding

classifier = load_ecapa()

In [10]:
utterance_df = pd.read_parquet(
    METADATA_DIR / "voxceleb_utterances.parquet"
)

print("Utterances:", len(utterance_df))
print("Speakers:", utterance_df["speaker_id"].nunique())

utterance_df.head()

Utterances: 124843
Speakers: 1166


,speaker_id,video_id,utterance_id,path,split,language,language_confidence
0,id10001,1zcIwhmdeo4,00001,/Users/juni/Documents/voxceleb1/wav/id10001/1z...,train,english,0.99
1,id10001,1zcIwhmdeo4,00003,/Users/juni/Documents/voxceleb1/wav/id10001/1z...,train,english,0.99
2,id10001,1zcIwhmdeo4,00002,/Users/juni/Documents/voxceleb1/wav/id10001/1z...,train,english,0.99
3,id10001,7gWzIy6yIIk,00004,/Users/juni/Documents/voxceleb1/wav/id10001/7g...,train,english,0.98
4,id10001,7gWzIy6yIIk,00001,/Users/juni/Documents/voxceleb1/wav/id10001/7g...,train,english,0.98


In [12]:
eligible_speakers = speaker_stats[
    speaker_stats["n_videos"] >= 5
].index.tolist()

print("Eligible speakers:", len(eligible_speakers))

Eligible speakers: 1132


In [13]:
import random

SEED = 42
N_SPEAKERS = 100
POS_PER_SPEAKER = 10
NEG_PER_SPEAKER = 10

rng = random.Random(SEED)

selected_speakers = rng.sample(
    eligible_speakers,
    N_SPEAKERS
)

len(selected_speakers)

100

In [14]:
positive_trials = []

for speaker in selected_speakers:

    speaker_df = utterance_df[
        utterance_df["speaker_id"] == speaker
    ]

    videos = speaker_df["video_id"].unique()

    for _ in range(POS_PER_SPEAKER):

        video1, video2 = rng.sample(list(videos), 2)

        utt1 = speaker_df[
            speaker_df["video_id"] == video1
        ].sample(
            1,
            random_state=rng.randint(0, 1_000_000)
        ).iloc[0]

        utt2 = speaker_df[
            speaker_df["video_id"] == video2
        ].sample(
            1,
            random_state=rng.randint(0, 1_000_000)
        ).iloc[0]

        positive_trials.append({
            "label": 1,
            "path1": utt1["path"],
            "path2": utt2["path"],
            "speaker1": speaker,
            "speaker2": speaker
        })

len(positive_trials)

1000

In [15]:
negative_trials = []

for _ in range(N_SPEAKERS * NEG_PER_SPEAKER):

    speaker1, speaker2 = rng.sample(
        selected_speakers,
        2
    )

    utt1 = utterance_df[
        utterance_df["speaker_id"] == speaker1
    ].sample(
        1,
        random_state=rng.randint(0, 1_000_000)
    ).iloc[0]

    utt2 = utterance_df[
        utterance_df["speaker_id"] == speaker2
    ].sample(
        1,
        random_state=rng.randint(0, 1_000_000)
    ).iloc[0]

    negative_trials.append({
        "label": 0,
        "path1": utt1["path"],
        "path2": utt2["path"],
        "speaker1": speaker1,
        "speaker2": speaker2
    })

len(negative_trials)

1000

In [16]:
verification_trials = pd.DataFrame(
    positive_trials + negative_trials
)

verification_trials = verification_trials.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

print(verification_trials.shape)
print()
print(verification_trials["label"].value_counts())

verification_trials.head()

(2000, 5)

label
0    1000
1    1000
Name: count, dtype: int64


,label,path1,path2,speaker1,speaker2
0,0,/Users/juni/Documents/voxceleb1/wav/id10819/aA...,/Users/juni/Documents/voxceleb1/wav/id10400/bu...,id10819,id10400
1,1,/Users/juni/Documents/voxceleb1/wav/id10095/kZ...,/Users/juni/Documents/voxceleb1/wav/id10095/E7...,id10095,id10095
2,0,/Users/juni/Documents/voxceleb1/wav/id10203/22...,/Users/juni/Documents/voxceleb1/wav/id10877/c_...,id10203,id10877
3,1,/Users/juni/Documents/voxceleb1/wav/id10198/Rw...,/Users/juni/Documents/voxceleb1/wav/id10198/Ep...,id10198,id10198
4,0,/Users/juni/Documents/voxceleb1/wav/id10540/5P...,/Users/juni/Documents/voxceleb1/wav/id10058/MV...,id10540,id10058


In [17]:
TRIALS_DIR = PROJECT_ROOT / "data" / "trials"
TRIALS_DIR.mkdir(parents=True, exist_ok=True)

OUR_TRIALS_PATH = (
    TRIALS_DIR / "voxceleb_verification_trials.csv"
)

verification_trials.to_csv(
    OUR_TRIALS_PATH,
    index=False
)

print("Saved:", OUR_TRIALS_PATH)

Saved: /Users/juni/Documents/interview-anonymization/data/trials/voxceleb_verification_trials.csv


In [18]:
unique_paths = sorted(
    set(verification_trials["path1"])
    | set(verification_trials["path2"])
)

print("Trials:", len(verification_trials))
print("Unique utterances:", len(unique_paths))

Trials: 2000
Unique utterances: 3100


In [19]:
from tqdm.auto import tqdm

embedding_cache = {}

for path in tqdm(unique_paths):
    embedding_cache[path] = extract_embedding(
        path,
        classifier
    )

print("Embeddings computed:", len(embedding_cache))

/opt/anaconda3/envs/interview-anon/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3100/3100 [07:00<00:00,  7.37it/s]

Embeddings computed: 3100


In [20]:
def cosine_similarity(z1, z2):
    z1 = np.asarray(z1).reshape(-1)
    z2 = np.asarray(z2).reshape(-1)

    z1 = z1 / np.linalg.norm(z1)
    z2 = z2 / np.linalg.norm(z2)

    return np.dot(z1, z2)

In [21]:
scores = []

for _, row in verification_trials.iterrows():
    z1 = embedding_cache[row["path1"]]
    z2 = embedding_cache[row["path2"]]

    score = cosine_similarity(z1, z2)
    scores.append(score)

verification_trials["score"] = scores

verification_trials[
    ["label", "speaker1", "speaker2", "score"]
].head(10)

,label,speaker1,speaker2,score
0,0,id10819,id10400,-0.063925
1,1,id10095,id10095,0.686856
2,0,id10203,id10877,0.210221
3,1,id10198,id10198,0.810739
4,0,id10540,id10058,0.005320
5,0,id10972,id10634,-0.046693
6,1,id10375,id10375,0.625903
7,0,id10068,id10500,0.100092
8,1,id10222,id10222,0.642701
9,0,id11159,id10401,0.114523


In [22]:
verification_trials.groupby("label")["score"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,1000.0,0.009359,0.090063,-0.329197,-0.049041,0.008544,0.063193,0.390092
1,1000.0,0.621444,0.116338,0.075330,0.554028,0.628478,0.698071,0.926370


In [27]:
from sklearn.metrics import roc_curve

labels = verification_trials["label"].to_numpy()
scores = verification_trials["score"].to_numpy()

fpr, tpr, thresholds = roc_curve(labels, scores)

fnr = 1 - tpr

eer_idx = np.nanargmin(np.abs(fpr - fnr))

eer = (fpr[eer_idx] + fnr[eer_idx]) / 2
eer_threshold = thresholds[eer_idx]

print(f"EER: {eer * 100:.20f}%")
print(f"EER threshold: {eer_threshold:.3f}")
print(f"FAR at EER: {fpr[eer_idx] * 100:.2f}%")
print(f"FRR at EER: {fnr[eer_idx] * 100:.2f}%")

EER: 1.00000000000000044409%
EER threshold: 0.240
FAR at EER: 1.10%
FRR at EER: 0.90%
